In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "paths.py").exists())
sys.path.insert(0, str(ROOT))
from paths import *


# OneTrainer on Colab - PixArt Sigma


In [ ]:
import os

PROJECT = str(ROOT)
REPO = f'{PROJECT}/OneTrainer'

os.makedirs(PROJECT, exist_ok=True)

if not os.path.exists(REPO):
    %cd $PROJECT
    !git clone --recursive https://github.com/Nerogar/OneTrainer.git

%cd $REPO

!rm -rf /content/OneTrainer /content/diffusers

!git log -1 --format="OneTrainer commit: %h  %cd"

print("cwd:", os.getcwd())


Mounted at /content/drive
/content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer
OneTrainer commit: 9a270603  Sun Feb 8 09:18:39 2026 +0100
Active dir: /content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer


In [ ]:

%cd {ROOT}/OneTrainer
!pip install -r requirements.txt


/content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu128
Ignoring triton-windows: markers 'sys_platform == "win32"' don't match your environment
Obtaining diffusers from git+https://github.com/huggingface/diffusers.git@6a1904e#egg=diffusers (from -r requirements-global.txt (line 23))
  Updating ./src/diffusers clone (to revision 6a1904e)
  Running command git fetch -q --tags
  Running command git reset --hard -q 6a1904e
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
Obtaining mgds from git+https://github.com/Nerogar/mgds.git@a0c84a3#egg=mgds (from -r requirements-global.txt (line 35))
  Updating ./src/mgds clone (to revision a0c84a3)
  Running command git fetch -q --tags
  Running command git reset --hard -q a0c84a3
  Installing build dependen

In [ ]:
!python -c "import torch, torchvision, transformers, diffusers; print('torch', torch.__version__); print('torchvision', torchvision.__version__); print('transformers', transformers.__version__); print('diffusers', diffusers.__version__)"
!python -c "import torch; print('cuda:', torch.cuda.is_available())"
!python -c "from torchvision.io import write_video; print('write_video OK')"
!python -c "import mgds; print('mgds:', mgds.__file__)"


torch 2.8.0+cu128
torchvision 0.23.0+cu128
transformers 4.56.2
diffusers 0.37.0.dev0
cuda: True
write_video OK
mgds: None


In [ ]:

import json, os

CONFIG = str(ROOT / "pixartsigma_colab.json")
REPO = str(ROOT / "OneTrainer")

c = json.load(open(CONFIG))

for k in ["base_model_name", "model_type", "training_method", "peft_type",
          "train_device", "temp_device", "resolution", "epochs", "batch_size",
          "learning_rate", "save_every", "save_every_unit",
          "workspace_dir", "output_model_destination"]:
    print(f"{k}: {c.get(k)}")

print()
problems = []

if c.get("concepts") == []:
    problems.append("concepts is [] -- must be null, or the concept file is ignored "
                    "and training silently runs on zero images")
if c.get("train_device") != "cuda":
    problems.append(f"train_device is {c.get('train_device')!r} -- should be 'cuda'")
if c.get("tensorboard"):
    problems.append("tensorboard is true -- Colab has no /usr/bin/tensorboard, set false")

cf = c.get("concept_file_name")
if cf:
    resolved = cf if os.path.isabs(cf) else os.path.join(REPO, cf)
    print("concept file:", resolved, "| exists:", os.path.exists(resolved))
    if os.path.exists(resolved):
        for con in json.load(open(resolved)):
            d = con.get("path")
            n = len(os.listdir(d)) if d and os.path.isdir(d) else 0
            print(f"  concept path: {d} | files: {n}")
            if n == 0:
                problems.append(f"concept path has no files: {d}")
    else:
        problems.append("concept file not found")

print()
if problems:
    for p in problems:
        print("PROBLEM:", p)
else:
    print("config looks OK")


base_model_name: PixArt-alpha/PixArt-Sigma-XL-2-1024-MS
model_type: PIXART_SIGMA
training_method: LORA
peft_type: LORA
train_device: cuda
temp_device: cpu
resolution: 1024
epochs: 100
batch_size: 4
learning_rate: 0.0001
save_every: 0
save_every_unit: NEVER
workspace_dir: /content/drive/MyDrive/Synthetic_Plants_Project/workspace/pixart_achillea_run
output_model_destination: /content/drive/MyDrive/Synthetic_Plants_Project/outputs/pixart_achillea/lora.safetensors

concept file: /content/drive/MyDrive/Synthetic_Plants_Project/Notebooks/OneTrainer/modelconfigs/train_concepts.json | exists: True
  concept path: /content/drive/MyDrive/Synthetic_Plants_Project/Datasets/Achillea_Maritima_2 | files: 276

config looks OK


## Train


In [ ]:

%%writefile /content/shim.py
import sys, os, runpy
import torchvision.io
if not hasattr(torchvision.io, "write_video"):
    torchvision.io.write_video = lambda *a, **k: None

root = os.getcwd()
sys.path.insert(0, os.path.join(root, "scripts"))
sys.path.insert(0, root)

sys.argv = sys.argv[1:]
runpy.run_path(os.path.join(root, "scripts", "train.py"), run_name="__main__")


Writing /content/shim.py


In [ ]:
%cd {ROOT}/OneTrainer
!python -u /content/shim.py train.py --config-path {ROOT}/training/configs/qwen/qwen_colab_achillea.json


Streaming output truncated to the last 5000 lines.
step:  72% 99/138 [03:43<01:26,  2.21s/it, loss=0.29, smooth loss=0.207] 
step:  72% 100/138 [03:43<01:25,  2.26s/it, loss=0.29, smooth loss=0.207]
step:  73% 101/138 [03:45<01:21,  2.20s/it, loss=0.29, smooth loss=0.207]
step:  74% 102/138 [03:47<01:18,  2.18s/it, loss=0.29, smooth loss=0.207]
step:  75% 103/138 [03:49<01:15,  2.15s/it, loss=0.29, smooth loss=0.207]
step:  75% 103/138 [03:51<01:15,  2.15s/it, loss=0.414, smooth loss=0.209]
step:  75% 104/138 [03:51<01:15,  2.22s/it, loss=0.414, smooth loss=0.209]
step:  76% 105/138 [03:54<01:16,  2.32s/it, loss=0.414, smooth loss=0.209]
step:  77% 106/138 [03:56<01:12,  2.26s/it, loss=0.414, smooth loss=0.209]
step:  78% 107/138 [03:58<01:08,  2.22s/it, loss=0.414, smooth loss=0.209]
step:  78% 107/138 [04:01<01:08,  2.22s/it, loss=0.127, smooth loss=0.209]
step:  78% 108/138 [04:01<01:07,  2.27s/it, loss=0.127, smooth loss=0.209]
step:  79% 109/138 [04:03<01:04,  2.21s/it, loss=0.127

In [ ]:
%cd {ROOT}/OneTrainer
!python -u /content/shim.py train.py --config-path {ROOT}/training/configs/qwen/qwen_colab_eryngium.json



step:  50% 45/90 [01:42<01:37,  2.16s/it, loss=0.125, smooth loss=0.217]
step:  51% 46/90 [01:42<01:43,  2.36s/it, loss=0.125, smooth loss=0.217]
step:  52% 47/90 [01:44<01:38,  2.28s/it, loss=0.125, smooth loss=0.217]
step:  53% 48/90 [01:47<01:40,  2.39s/it, loss=0.125, smooth loss=0.217]
step:  54% 49/90 [01:49<01:34,  2.31s/it, loss=0.125, smooth loss=0.217]
step:  54% 49/90 [01:52<01:34,  2.31s/it, loss=0.474, smooth loss=0.22] 
step:  56% 50/90 [01:52<01:33,  2.33s/it, loss=0.474, smooth loss=0.22]
step:  57% 51/90 [01:54<01:27,  2.25s/it, loss=0.474, smooth loss=0.22]
step:  58% 52/90 [01:56<01:24,  2.23s/it, loss=0.474, smooth loss=0.22]
step:  59% 53/90 [01:58<01:21,  2.20s/it, loss=0.474, smooth loss=0.22]
step:  59% 53/90 [02:00<01:21,  2.20s/it, loss=0.176, smooth loss=0.219]
step:  60% 54/90 [02:00<01:21,  2.25s/it, loss=0.176, smooth loss=0.219]
step:  61% 55/90 [02:02<01:17,  2.20s/it, loss=0.176, smooth loss=0.219]
step:  62% 56/90 [02:04<01:13,  2.18s/it, loss=0.176, 

In [9]:
%cd {ROOT}/OneTrainer
!python -u /content/shim.py train.py --config-path {ROOT}/training/configs/qwen/qwen_colab_carpobrotus.json



step:  96% 67/70 [02:28<00:06,  2.23s/it, loss=0.239, smooth loss=0.216]
step:  96% 67/70 [02:30<00:06,  2.23s/it, loss=0.194, smooth loss=0.216]
step:  97% 68/70 [02:30<00:04,  2.27s/it, loss=0.194, smooth loss=0.216]
step:  99% 69/70 [02:32<00:02,  2.22s/it, loss=0.194, smooth loss=0.216]
step: 100% 70/70 [02:35<00:00,  2.22s/it, loss=0.194, smooth loss=0.216]
epoch:  89% 89/100 [4:18:49<29:59, 163.55s/it]
step:   0% 0/70 [00:00<?, ?it/s]
step:   1% 1/70 [00:02<02:27,  2.14s/it]
step:   1% 1/70 [00:04<02:27,  2.14s/it, loss=0.496, smooth loss=0.219]
step:   3% 2/70 [00:04<02:34,  2.27s/it, loss=0.496, smooth loss=0.219]
step:   4% 3/70 [00:06<02:25,  2.18s/it, loss=0.496, smooth loss=0.219]
step:   6% 4/70 [00:08<02:21,  2.15s/it, loss=0.496, smooth loss=0.219]
step:   7% 5/70 [00:10<02:18,  2.14s/it, loss=0.496, smooth loss=0.219]
step:   7% 5/70 [00:13<02:18,  2.14s/it, loss=0.123, smooth loss=0.218]
step:   9% 6/70 [00:13<02:22,  2.22s/it, loss=0.123, smooth loss=0.218]
step:  10

In [10]:
%cd {ROOT}
for s in ["carpobrotus"]:
    !python Configs_and_Concepts/generate_eval_qwen.py --species {s} --n 100


/content/drive/MyDrive/Synthetic_Plants_Project
model=qwen  n=100  species: carpobrotus

qwen/carpobrotus
  prompt: a detailed realistic photograph of z47mfpwjhs
  size:   1024x1024  steps=20  true_cfg=4.0
  out:    /content/drive/MyDrive/Synthetic_Plants_Project/generated/qwen_carpobrotus
2026-08-27 22:34:20.429777: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-27 22:34:20.505982: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend m

In [ ]:
ls -la {ROOT}/training/concepts/qwen


total 1065
-rw------- 1 root root 510264 Aug 27 17:17 'Copy of QWEN_OneTrainer_Colab_clean (2).ipynb'
-rw------- 1 root root  14950 Aug 26 21:28  qwen_colab_achillea.json
-rw------- 1 root root  14965 Aug 26 21:28  qwen_colab_carpobrotus.json
-rw------- 1 root root  14950 Aug 26 21:28  qwen_colab_eryngium.json
-rw------- 1 root root 507218 Aug 27 17:23 'QWEN_OneTrainer_Colab_clean (2).ipynb'
-rw------- 1 root root   2254 Aug 26 21:30  train_concepts_achillea.json
-rw------- 1 root root   2270 Aug 26 21:30  train_concepts_carpobrotus.json
-rw------- 1 root root   2256 Aug 26 21:30  train_concepts_eryngium.json
-rw------- 1 root root  15337 Aug 26 21:22  train_config5.json
-rw------- 1 root root    832 Aug 26 21:30  train_samples_achillea.json
-rw------- 1 root root    832 Aug 27 17:23 'train_samples_carpobrotus (5).json'
-rw------- 1 root root    832 Aug 26 21:30  train_samples_eryngium.json


In [ ]:
%cd {ROOT}/OneTrainer
!python -u /content/shim.py train.py --config-path {ROOT}/training/configs/pixart/pixartsigma_colab_carpobrotus.json


In [ ]:
%cd {ROOT}/OneTrainer
!python -u /content/shim.py train.py --config-path {ROOT}/training/configs/pixart/pixartsigma_colab_carpobrotus.json


In [ ]:
%cd {ROOT}
!python Configs_and_Concepts/generate_eval.py --all --n 100


In [ ]:
# @title
%cd {ROOT}
!python Configs_and_Concepts/generate_eval.py --species achillea --n 2


/content/drive/MyDrive/Synthetic_Plants_Project
2026-08-26 20:52:04.171909: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-26 20:52:04.243580: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
generating 2 per species: ac

In [ ]:
# @title
from safetensors import safe_open
p = str(ROOT / "outputs/lora/pixart_achillea/lora.safetensors")
with safe_open(p, framework="pt") as f:
    keys = list(f.keys())
print(len(keys), "tensors")
for k in keys[:12]:
    print(" ", k)


861 tensors
  lora_transformer_adaln_single_emb_timestep_embedder_linear_1.alpha
  lora_transformer_adaln_single_emb_timestep_embedder_linear_1.lora_down.weight
  lora_transformer_adaln_single_emb_timestep_embedder_linear_1.lora_up.weight
  lora_transformer_adaln_single_emb_timestep_embedder_linear_2.alpha
  lora_transformer_adaln_single_emb_timestep_embedder_linear_2.lora_down.weight
  lora_transformer_adaln_single_emb_timestep_embedder_linear_2.lora_up.weight
  lora_transformer_adaln_single_linear.alpha
  lora_transformer_adaln_single_linear.lora_down.weight
  lora_transformer_adaln_single_linear.lora_up.weight
  lora_transformer_caption_projection_linear_1.alpha
  lora_transformer_caption_projection_linear_1.lora_down.weight
  lora_transformer_caption_projection_linear_1.lora_up.weight


In [ ]:
# @title
# !ls -la {ROOT}/outputs/workspace*/backup/

# %cd {ROOT}/OneTrainer
# !python -u /content/shim.py train.py \
#   --config-path {ROOT}/training/configs/pixart/pixartsigma_colab.json \
#   --resume-from-checkpoint {ROOT}/outputs/workspace<run>/backup/last


In [ ]:
# @title
from datetime import datetime
stamp = datetime.now().strftime("%Y%m%d_%H%M")
!pip freeze > {ROOT}/requirements_frozen_{stamp}.txt
print("written: requirements_frozen_" + stamp + ".txt")


written: requirements_frozen_20260825_0018.txt
